# 09 — Error and Event Analysis

**Purpose:** Explain **why** the frozen NB08 aggregate metrics look the way they do — moving from point/sequence-level metrics to false-positive patterns, false-negative patterns, attack-event behaviour, event coverage, detection delay, and TCN-vs-LSTM complementarity.

**No model is trained, tuned, retrained, threshold-calibrated, or altered in this notebook.** All scores, thresholds, and sequence indices are loaded read-only from `nb08_artifacts\`. No write touches `nb08_artifacts\`.

## Prerequisite gates
- **GAP 1 (`List_of_attacks_Final.xlsx`)** must be present for Phases 5 and 6 (event-level) to run. If absent, Cells 5–6 set `HAVE_SCENARIOS=False` and Phases 5–6 event-level cells print BLOCKED and skip. **Phases 2–4 and point-level Phase 6 still run.**
- **GAP 2 (derived normal-partition timestamps)** is resolved by Cell 4. This must succeed before any temporal analysis.

## How to read this notebook
- **Phase 2** reconstructs TP/FP/TN/FN at the frozen thresholds and **asserts** equality with `evaluation_summary.csv`. If this fails, the notebook stops — nothing downstream is valid.
- **Phase 3 / Phase 4** characterise FPs and FNs at sequence level (clustering by 300 s gap, per-sensor attribution).
- **Phase 5** (gated) computes per-scenario detection, coverage, and delay for all 36 SUTD scenarios.
- **Phase 6** tests whether TCN-AE and LSTM-AE errors are **complementary**. Cell 26 is **descriptive only** — it does **not** select an ensemble.

In [2]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

In [ ]:
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                              precision_recall_curve, auc)
from scipy.stats import spearmanr
import joblib

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path(__file__).resolve().parent
DATA_DIR = PROJECT_ROOT / "swat-dataset"
NB08_DIR = PROJECT_ROOT / "nb08_artifacts"
NB09_OUT_DIR = PROJECT_ROOT / "nb09_analysis"
NB09_OUT_DIR.mkdir(parents=True, exist_ok=True)


CHECKSUM_BASELINE = PROJECT_ROOT / "_nb08_checksums_BASELINE.csv"
ATTACK_LIST_PATH  = DATA_DIR / "List_of_attacks_Final.xlsx"
NORMAL_CSV        = DATA_DIR / "normal.csv"
ATTACK_CSV        = DATA_DIR / "attack.csv"

METHODS     = ["rule","rolling","simple","if","ocsvm","lof","lstm","tcn","usad"]
OBS_METHODS = ["rule","rolling","simple","if","ocsvm","lof"]

assert NB08_DIR.exists(), f"nb08_artifacts missing at {NB08_DIR}"
assert CHECKSUM_BASELINE.exists(), f"checksum baseline missing at {CHECKSUM_BASELINE}"
assert NB09_OUT_DIR.exists()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"NB09_OUT_DIR = {NB09_OUT_DIR}")
print(f"METHODS = {METHODS}")